# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShubhamSnSharma/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

We examine two core empirical findings from FlyRank's March 2026 research paper (*The State of AI-Driven SEO in Numbers*), analyzing the study design, underlying assumptions, and methodological questions:

---

### Finding 1: #4, The Freshness Multiplier
- **Paper Finding**: *"365+ day content that was refreshed within 30 days shows a 3.2x health boost (10.7 to 34.5) and 57x more impressions (71 to 4039). Freshness multiplies existing quality."* *(Note: Finding #8 provides supporting context on the 365+ rebound cohort.)*
- **Methodology Questions (Study Design & Assumptions)**:
  - *What outcome and comparison does the finding use?* The analysis compares historical 90-day GSC impressions and FlyRank's composite `health_score` across content age cohorts against `days_since_last_update` buckets.
  - *What assumptions are required for the reported conclusion?* The comparison assumes that refreshed mature pages are otherwise comparable to untouched mature pages. However, in practice, human editorial teams selectively choose high-priority, commercially valuable, or historically authoritative pages to refresh (non-random selection). Furthermore, filtering on the active-content subset (`impressions > 0`, `sessions > 0`) omits pages that decayed completely, introducing survivorship bias.
  - *Constructive Framing*: The observational correlation is valuable for **decision-support candidate ranking** (identifying mature pages that may be candidates for further review), but proving a causal multiplier requires randomized A/B rollouts or difference-in-differences testing.

---

### Finding 2: #10, AI-Generated Content Performance
- **Paper Finding**: *"Within this mostly AI-authored portfolio, age-controlled model cohorts do not show a simple blanket penalty tied only to AI use... Gemini leads some cohorts and OpenAI leads others."*
- **Methodology Questions (Study Design & Assumptions)**:
  - *What outcome and comparison does the finding use?* The study evaluates observed search performance across content pieces attributed to different LLM providers (`provider_used`, `model_used`) across age cohorts.
  - *What assumptions are required for the reported conclusion?* Comparing providers assumes that content generation workflows, topic difficulty, prompt templates, and client domain authority are evenly distributed across models. In reality, client accounts adopted specific LLM models for specific programmatic formats and at different times. Additionally, because the dataset observes indexed pages, it does not account for whether different models experienced differential non-indexing rates (as a substantial share of pages across the portfolio never reached indexed search performance).
  - *Constructive Framing*: Observed differences suggest that workflow standards, human editorial review, keyword selection, and other contextual factors may matter alongside LLM provider identity when interpreting search visibility.

In [10]:
import pandas as pd

# Structured summary matrix of research paper audit
audit_summary = pd.DataFrame([
    {
        "Finding": "#4: The Freshness Multiplier",
        "Reported Comparison": "3.2x health score lift & 57x impressions for refreshed 365+d pages",
        "Outcome Definition": "GSC 90d impressions + composite health_score",
        "Methodological Question": "Non-random editorial selection bias & active-subset survivorship bias",
        "Constructive Framing": "Useful for decision-support triage; requires A/B holdouts for causal proof"
    },
    {
        "Finding": "#10: AI-Generated Content Performance",
        "Reported Comparison": "Performance variation across LLM models across age cohorts",
        "Outcome Definition": "dim_content metadata joined with GSC performance",
        "Methodological Question": "Confounding by client domain authority, prompt pipeline, and unindexed rate",
        "Constructive Framing": "Search outcomes reflect topic & editorial quality alongside LLM provider"
    }
])

print("=== Summary of Research Paper Methodology Audit ===")
display(audit_summary)

=== Summary of Research Paper Methodology Audit ===


,Finding,Reported Comparison,Outcome Definition,Methodological Question,Constructive Framing
0,#4: The Freshness Multiplier,3.2x health score lift & 57x impressions for r...,GSC 90d impressions + composite health_score,Non-random editorial selection bias & active-s...,Useful for decision-support triage; requires A...
1,#10: AI-Generated Content Performance,Performance variation across LLM models across...,dim_content metadata joined with GSC performance,"Confounding by client domain authority, prompt...",Search outcomes reflect topic & editorial qual...


## 2. My model under an honest split (before/after)

### Auditing the W05 Validation Strategy

In **Week 5 (`w05_model.ipynb`)**, our official submission trained a standardized **Logistic Regression** model on 9 static features using a client-grouped holdout split (`GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`).

#### The Genuine Validation Vulnerability in W05
While W05 correctly used a client-grouped split (avoiding client leakage across train and test), evaluating on **a single 8-client test fold** introduces sample variance: ranking only the top 50 pages out of a single client partition produces a point estimate that is sensitive to the particular client mix. Furthermore, although each split holds out 8 clients, the number of pages varies substantially because client portfolio sizes differ.

#### Our Enhanced Validation Design (Before vs. After)
1. **Before (W05 Original)**: Single client-holdout split (`GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`).
2. **After (5 Repeated GroupShuffleSplit Evaluations)**: Evaluate both the Baseline and Logistic Regression across **5 repeated client-grouped splits** (`test_size=0.20`, deterministic seeds 42, 43, 44, 45, 46). We compute Precision@50 **separately within each held-out split**, reporting per-split scores, Mean $\pm$ Std, Min, and Max. The repeated client-grouped evaluation provides a broader view of performance across held-out client groups in this dataset.
3. **Diagnostic Comparison (Naive Random Split)**: We also run a naive row-level random split (`ShuffleSplit`) strictly as a diagnostic tool to illustrate why allowing pages from the same client into both train and test can produce a more optimistic estimate.

In [11]:
import os, datetime, duckdb, numpy as np, pandas as pd
from getpass import getpass
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# -------------------------------------------------------------------------
# Step 1: Secure Authentication & Warehouse Data Loading (W05 Exact Query)
# -------------------------------------------------------------------------
token = os.getenv("HF_TOKEN")
if not token:
    try:
        token = getpass("Enter your Hugging Face READ token: ")
    except Exception:
        token = None

if not token:
    raise ValueError("HF_TOKEN environment variable not set. Please provide a valid Hugging Face token.")

con = duckdb.connect()
con.execute(f"CREATE SECRET IF NOT EXISTS hf_s (TYPE huggingface, TOKEN '{token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Query fact_content_daily_performance with explicit pre-May metadata windows
w05_sql = f"""
WITH daily_facts AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        gsc_avg_position,
        COALESCE(ga4_pageviews, 0) AS ga4_pageviews,
        COALESCE(ga4_engaged_sessions, 0) AS ga4_engaged_sessions,
        COALESCE(client_has_gsc, FALSE) AS client_has_gsc,
        COALESCE(client_has_ga4, FALSE) AS client_has_ga4,
        COALESCE(gsc_data_available, FALSE) AS gsc_data_available,
        COALESCE(ga4_data_available, FALSE) AS ga4_data_available
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month IN ('2026-02', '2026-03', '2026-04', '2026-05')
),
aggregated_pages AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature window aggregations (Feb - Apr 2026)
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_impressions ELSE 0 END) AS gsc_impressions_feb_apr,
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_clicks ELSE 0 END) AS gsc_clicks_feb_apr,
        SUM(CASE WHEN month = '2026-04' THEN gsc_impressions ELSE 0 END) AS gsc_impressions_apr,
        SUM(CASE WHEN month = '2026-04' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_apr,
        AVG(CASE WHEN month = '2026-04' THEN gsc_avg_position ELSE NULL END) AS gsc_avg_position_apr,

        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_pageviews ELSE 0 END) AS ga4_pageviews_feb_apr,
        SUM(CASE WHEN month = '2026-04' THEN ga4_pageviews ELSE 0 END) AS ga4_pageviews_apr,
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions_feb_apr,

        -- Explicitly bounded pre-May metadata & data availability flags
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN client_has_gsc END) AS client_has_gsc,
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN client_has_ga4 END) AS client_has_ga4,
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_data_available END) AS gsc_data_available,
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_data_available END) AS ga4_data_available,

        -- Target window aggregation (May 2026) - ISOLATED FOR TARGET ONLY
        SUM(CASE WHEN month = '2026-05' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_may

    FROM daily_facts
    GROUP BY client_hash_id, content_hash_id
)
SELECT *
FROM aggregated_pages
WHERE gsc_impressions_feb_apr >= 1000
  AND gsc_clicks_apr >= 10
"""

modeling_df = con.sql(w05_sql).df()
modeling_df = modeling_df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

# Compute Week 4 baseline score (daily average across April)
baseline_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(
        (
            CASE WHEN COALESCE(gsc_impressions,0) >= 10 THEN 1 ELSE 0 END +
            CASE WHEN gsc_avg_position > 10 THEN 1 ELSE 0 END +
            CASE WHEN COALESCE(ga4_pageviews,0) >= 1 THEN 1 ELSE 0 END
        )
    ) AS baseline_score
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-04'
GROUP BY
    client_hash_id,
    content_hash_id
"""
baseline_df = con.sql(baseline_sql).df()
modeling_df = modeling_df.merge(baseline_df, on=["client_hash_id", "content_hash_id"], how="left")
modeling_df["baseline_score"] = modeling_df["baseline_score"].fillna(0.0)

# Target definition (May 2026 decay >= 20%)
modeling_df["is_declining_label"] = (
    modeling_df["gsc_clicks_may"] < (0.80 * modeling_df["gsc_clicks_apr"])
).astype(int)
modeling_df["gsc_avg_position_apr"] = modeling_df["gsc_avg_position_apr"].fillna(0.0)

# Feature engineering (W05 official features)
modeling_df["log_gsc_impressions_feb_apr"] = np.log1p(modeling_df["gsc_impressions_feb_apr"])
modeling_df["log_gsc_clicks_apr"] = np.log1p(modeling_df["gsc_clicks_apr"])
modeling_df["log_ga4_pageviews_feb_apr"] = np.log1p(modeling_df["ga4_pageviews_feb_apr"])
modeling_df["log_ga4_engaged_sessions_feb_apr"] = np.log1p(modeling_df["ga4_engaged_sessions_feb_apr"])

feature_cols = [
    "log_gsc_impressions_feb_apr",
    "log_gsc_clicks_apr",
    "gsc_avg_position_apr",
    "log_ga4_pageviews_feb_apr",
    "log_ga4_engaged_sessions_feb_apr",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

X = modeling_df[feature_cols].astype(float)
y = modeling_df["is_declining_label"].astype(int)
groups = modeling_df["client_hash_id"].astype(str)

# Evaluation helper functions with deterministic tie-breaking
def precision_at_k(y_true, scores, tie_df, k=50):
    eval_df = tie_df.copy()
    eval_df["y"] = list(y_true)
    eval_df["score"] = list(scores)
    top = eval_df.sort_values(
        ["score", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
        ascending=[False, False, False, True]
    ).head(k)
    return float(top["y"].mean()), int(top["y"].sum())

def baseline_precision_at_k(test_subset, k=50):
    top = test_subset.sort_values(
        ["baseline_score", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
        ascending=[False, False, False, True]
    ).head(k)
    return float(top["is_declining_label"].mean()), int(top["is_declining_label"].sum())

print("=== Dataset Validation Summary ===")
print(f"Eligible Pages (N): {len(modeling_df):,} across {groups.nunique()} clients")
print(f"Target class balance: {y.value_counts().to_dict()} (Declining proportion: {y.mean():.4f})")

# -------------------------------------------------------------------------
# 1. Original W05 Single Split Validation (GroupShuffleSplit, seed=42)
# -------------------------------------------------------------------------
gss_orig = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss_orig.split(X, y, groups=groups))

X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
test_df = modeling_df.iloc[te_idx].copy()

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
])
lr_pipeline.fit(X_tr, y_tr)
test_probs = lr_pipeline.predict_proba(X_te)[:, 1]

orig_base_p50, orig_base_tp = baseline_precision_at_k(test_df, k=50)
orig_lr_p50, orig_lr_tp = precision_at_k(y_te, test_probs, test_df, k=50)

print("\n--- 1. W05 Original Single Split Results (Official 'Before' Result) ---")
print(f"Train: {len(X_tr):,} rows ({groups.iloc[tr_idx].nunique()} clients) | Test: {len(X_te):,} rows ({groups.iloc[te_idx].nunique()} clients)")
print(f"Week 4 Baseline Rule Precision@50: {orig_base_p50:.4f} ({orig_base_tp}/50)")
print(f"Logistic Regression  Precision@50: {orig_lr_p50:.4f} ({orig_lr_tp}/50)")
print(f"Measured Single Split Difference: {orig_lr_p50 - orig_base_p50:+.4f}")

# -------------------------------------------------------------------------
# 2. Enhanced Validation: 5 Repeated GroupShuffleSplit Evaluations (seeds 42-46)
# -------------------------------------------------------------------------
split_seeds = [42, 43, 44, 45, 46]
split_records = []

for seed in split_seeds:
    s_gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    s_tr_idx, s_te_idx = next(s_gss.split(X, y, groups=groups))

    s_X_tr, s_X_te = X.iloc[s_tr_idx], X.iloc[s_te_idx]
    s_y_tr, s_y_te = y.iloc[s_tr_idx], y.iloc[s_te_idx]
    s_test_df = modeling_df.iloc[s_te_idx].copy()

    s_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
    ])
    s_pipeline.fit(s_X_tr, s_y_tr)
    s_test_probs = s_pipeline.predict_proba(s_X_te)[:, 1]

    s_base_p50, s_base_tp = baseline_precision_at_k(s_test_df, k=50)
    s_lr_p50, s_lr_tp = precision_at_k(s_y_te, s_test_probs, s_test_df, k=50)

    split_records.append({
        "Seed": seed,
        "Held-Out Pages": len(s_test_df),
        "Held-Out Clients": groups.iloc[s_te_idx].nunique(),
        "Baseline P@50": s_base_p50,
        "Baseline TP": f"{s_base_tp}/50",
        "LR P@50": s_lr_p50,
        "LR TP": f"{s_lr_tp}/50",
        "Lift": f"{s_lr_p50 - s_base_p50:+.4f}"
    })

split_results = pd.DataFrame(split_records)
print("\n--- 2. Enhanced 5 Repeated GroupShuffleSplit Results ---")
display(split_results)

calc_base_p50 = split_results["Baseline P@50"].astype(float)
calc_lr_p50 = split_results["LR P@50"].astype(float)
lift_values = calc_lr_p50 - calc_base_p50

print(f"\nBaseline 5-Split P@50 -> Mean: {calc_base_p50.mean():.4f} ± {calc_base_p50.std():.4f} (Min: {calc_base_p50.min():.4f}, Max: {calc_base_p50.max():.4f})")
print(f"LR 5-Split P@50       -> Mean: {calc_lr_p50.mean():.4f} ± {calc_lr_p50.std():.4f} (Min: {calc_lr_p50.min():.4f}, Max: {calc_lr_p50.max():.4f})")
print(f"Mean Split-Level Lift : {lift_values.mean():+.4f} ± {lift_values.std():.4f} (Min: {lift_values.min():+.4f}, Max: {lift_values.max():+.4f})")

wins = int((lift_values > 0).sum())
losses = int((lift_values < 0).sum())
ties = int((lift_values == 0).sum())

print(f"\nValidation Finding:")
print(f"The repeated client-grouped evaluation shows substantial variation across client partitions. Logistic Regression outperformed the baseline in {wins} of {len(split_seeds)} splits, underperformed it in {losses}, and tied in {ties}.")
print(f"The mean split-level lift was {lift_values.mean()*100:+.1f} percentage points, with a range from {lift_values.min()*100:+.1f} to {lift_values.max()*100:+.1f} percentage points.")
print("This means the +4.0 percentage-point improvement observed in the original W05 split should not be treated as a stable improvement across client partitions.")
print("Although each split holds out 8 clients, the number of pages varies substantially because client sizes differ. Precision@50 is therefore sensitive to the particular client mix in each split.")

# -------------------------------------------------------------------------
# 3. Diagnostic Check: Naive Random Row Split (Memorization Diagnostic)
# -------------------------------------------------------------------------
ss = ShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
r_tr_idx, r_te_idx = next(ss.split(X, y))

r_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
])
r_pipeline.fit(X.iloc[r_tr_idx], y.iloc[r_tr_idx])
r_test_probs = r_pipeline.predict_proba(X.iloc[r_te_idx])[:, 1]
r_lr_p50, r_lr_tp = precision_at_k(y.iloc[r_te_idx], r_test_probs, modeling_df.iloc[r_te_idx], k=50)

diff_pp = (r_lr_p50 - orig_lr_p50) * 100.0
print("\n--- 3. Diagnostic Check: Naive Random Row Split ---")
print(f"In this diagnostic split, the naive row-level split produced Precision@50 = {r_lr_p50:.4f}, which was {diff_pp:+.1f} percentage points higher than the W05 client-holdout result of {orig_lr_p50:.4f}. The difference illustrates why allowing pages from the same client into both train and test can produce a more optimistic estimate.")

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Dataset Validation Summary ===
Eligible Pages (N): 16,513 across 36 clients
Target class balance: {0: 9640, 1: 6873} (Declining proportion: 0.4162)

--- 1. W05 Original Single Split Results (Official 'Before' Result) ---
Train: 14,786 rows (28 clients) | Test: 1,727 rows (8 clients)
Week 4 Baseline Rule Precision@50: 0.2000 (10/50)
Logistic Regression  Precision@50: 0.2400 (12/50)
Measured Single Split Difference: +0.0400

--- 2. Enhanced 5 Repeated GroupShuffleSplit Results ---


,Seed,Held-Out Pages,Held-Out Clients,Baseline P@50,Baseline TP,LR P@50,LR TP,Lift
0,42,1727,8,0.20,10/50,0.24,12/50,+0.0400
1,43,6324,8,0.14,7/50,0.28,14/50,+0.1400
2,44,4763,8,0.56,28/50,0.44,22/50,-0.1200
3,45,802,8,0.56,28/50,0.48,24/50,-0.0800
4,46,3796,8,0.16,8/50,0.26,13/50,+0.1000



Baseline 5-Split P@50 -> Mean: 0.3240 ± 0.2165 (Min: 0.1400, Max: 0.5600)
LR 5-Split P@50       -> Mean: 0.3400 ± 0.1114 (Min: 0.2400, Max: 0.4800)
Mean Split-Level Lift : +0.0160 ± 0.1126 (Min: -0.1200, Max: +0.1400)

Validation Finding:
The repeated client-grouped evaluation shows substantial variation across client partitions. Logistic Regression outperformed the baseline in 3 of 5 splits, underperformed it in 2, and tied in 0.
The mean split-level lift was +1.6 percentage points, with a range from -12.0 to +14.0 percentage points.
This means the +4.0 percentage-point improvement observed in the original W05 split should not be treated as a stable improvement across client partitions.
Although each split holds out 8 clients, the number of pages varies substantially because client sizes differ. Precision@50 is therefore sensitive to the particular client mix in each split.

--- 3. Diagnostic Check: Naive Random Row Split ---
In this diagnostic split, the naive row-level split produc

## 3. Leakage audit

We audit the **9 official W05 production features** against the three core leakage failure modes from `hunting-leakage-and-validating`:

1. **Label-Derived Features**: The prediction target is May 2026 organic decay (`May clicks < 0.80 * April clicks`). We confirm that no May search data, post-prediction click ratios, or target proxies exist in the feature set.
2. **Future / Overlapping Windows**: All features strictly use February 1, 2026 – April 30, 2026 data. The target window is strictly May 1, 2026 – May 31, 2026. Zero day overlap.
3. **Decision-Derived Features**: No product triage flags, `health_score`, or priority classifications are included in `X`.

We also run an executable **controlled leakage demonstration** by deliberately modifying the feature matrix to include future May clicks (`gsc_clicks_may`), retraining the pipeline on the training split, predicting on the held-out test split, and calculating actual metrics.

In [12]:
from sklearn.metrics import roc_auc_score

# -------------------------------------------------------------------------
# 1. Feature-by-Feature Temporal and Pre-Decision Audit Table
# -------------------------------------------------------------------------
w05_leakage_audit = pd.DataFrame([
    {"Feature": "log_gsc_impressions_feb_apr", "Window": "Feb-Apr 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Historical)"},
    {"Feature": "log_gsc_clicks_apr", "Window": "April 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Historical)"},
    {"Feature": "gsc_avg_position_apr", "Window": "April 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Historical)"},
    {"Feature": "log_ga4_pageviews_feb_apr", "Window": "Feb-Apr 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Historical)"},
    {"Feature": "log_ga4_engaged_sessions_feb_apr", "Window": "Feb-Apr 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Historical)"},
    {"Feature": "client_has_gsc", "Window": "Feb-Apr 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Contextual)"},
    {"Feature": "client_has_ga4", "Window": "Feb-Apr 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Contextual)"},
    {"Feature": "gsc_data_available", "Window": "Feb-Apr 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Contextual)"},
    {"Feature": "ga4_data_available", "Window": "Feb-Apr 2026", "Pre-Decision?": "Yes", "Overlaps Target?": "No", "Encodes Target?": "No", "Leakage Decision": "Clean (Contextual)"}
])

print("=== W05 Feature Set Leakage Audit Table ===")
display(w05_leakage_audit)

# -------------------------------------------------------------------------
# 2. Controlled Leakage Demonstration: Injecting May 2026 Clicks Live
# -------------------------------------------------------------------------
X_leaky = X.copy()
X_leaky["gsc_clicks_may"] = modeling_df["gsc_clicks_may"].astype(float)

leaky_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
])

leaky_pipeline.fit(X_leaky.iloc[tr_idx], y_tr)
leaky_probs = leaky_pipeline.predict_proba(X_leaky.iloc[te_idx])[:, 1]

leaky_p50, leaky_tp = precision_at_k(y_te, leaky_probs, test_df, k=50)
leaky_auc = roc_auc_score(y_te, leaky_probs)
clean_auc = roc_auc_score(y_te, test_probs)

print("\n=== Controlled Leakage Demonstration (Executable) ===")
print(f"Clean W05 Model  -> Precision@50: {orig_lr_p50:.4f} ({orig_lr_tp}/50) | ROC-AUC: {clean_auc:.4f}")
print(f"Leaky Model (May)-> Precision@50: {leaky_p50:.4f} ({leaky_tp}/50) | ROC-AUC: {leaky_auc:.4f}")
print(f"The controlled injection produced a large performance increase, demonstrating how including future May information can materially distort the evaluation.")

=== W05 Feature Set Leakage Audit Table ===


,Feature,Window,Pre-Decision?,Overlaps Target?,Encodes Target?,Leakage Decision
0,log_gsc_impressions_feb_apr,Feb-Apr 2026,Yes,No,No,Clean (Historical)
1,log_gsc_clicks_apr,April 2026,Yes,No,No,Clean (Historical)
2,gsc_avg_position_apr,April 2026,Yes,No,No,Clean (Historical)
3,log_ga4_pageviews_feb_apr,Feb-Apr 2026,Yes,No,No,Clean (Historical)
4,log_ga4_engaged_sessions_feb_apr,Feb-Apr 2026,Yes,No,No,Clean (Historical)
5,client_has_gsc,Feb-Apr 2026,Yes,No,No,Clean (Contextual)
6,client_has_ga4,Feb-Apr 2026,Yes,No,No,Clean (Contextual)
7,gsc_data_available,Feb-Apr 2026,Yes,No,No,Clean (Contextual)
8,ga4_data_available,Feb-Apr 2026,Yes,No,No,Clean (Contextual)



=== Controlled Leakage Demonstration (Executable) ===
Clean W05 Model  -> Precision@50: 0.2400 (12/50) | ROC-AUC: 0.4594
Leaky Model (May)-> Precision@50: 1.0000 (50/50) | ROC-AUC: 0.8921
The controlled injection produced a large performance increase, demonstrating how including future May information can materially distort the evaluation.


## 4. Claim rewrite

We inspect our model's predictions on the held-out test set to perform an error analysis, then rewrite modeling claims using the **Claim Ladder** (`observed -> directional -> decision-support`, never causal without a controlled experiment).

---

### Claim Ladder Rewrite Table

| Original Overreaching / Naive Claim | Why It Fails | Rewritten Public-Safe Claim |
|---|---|---|
| *"Logistic Regression proves that optimizing static search factors prevents 24% of page traffic decay."* | Confuses prediction with causation; cross-sectional observational data cannot prove editorial intervention lift without an A/B test. | *"On the original W05 client-held-out split, Logistic Regression achieved Precision@50 = 0.2400 compared with 0.2000 for the Week 4 baseline. Across the repeated client-grouped validation used in this audit, performance varied across client partitions, so the single-split difference should be treated as a measured result for that split rather than a stable general improvement."* |
| *"The machine learning model predicted Google's algorithm ranking factors."* | No model predicts Google's proprietary search algorithms; the model only learns empirical associations with page-level traffic changes across 36 client domains. | *"The model learned statistical associations between historical search performance, GA4 engagement, average position, and the subsequent May click-decline label within this dataset."* |
| *"Logistic Regression achieved high accuracy on SEO data."* | Citing accuracy masks the base rate (41.6%) and test-set generalization; model utility is determined by top-K ranking precision. | *"Evaluated using Precision@50 for decision-support ranking, Logistic Regression improved top-50 precision from 0.2000 to 0.2400 on the original client-held-out split."* |

In [13]:
# -------------------------------------------------------------------------
# Real Held-Out Error Analysis on Actual W05 Predictions
# -------------------------------------------------------------------------
pred_df = test_df.copy()
pred_df["predicted_probability"] = test_probs

# Sort deterministically by model score and tie-breaking hierarchy
top50_df = pred_df.sort_values(
    ["predicted_probability", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
    ascending=[False, False, False, True]
).head(50)

true_positives = top50_df[top50_df["is_declining_label"] == 1]
false_positives = top50_df[top50_df["is_declining_label"] == 0]

print("=== W05 Logistic Regression Top-50 Queue Breakdown (Test Set) ===")
print(f"Total Reviewed: {len(top50_df)}")
print(f"True Positives (Successfully Flagged Declining Pages): {len(true_positives)} ({len(true_positives)/len(top50_df)*100:.1f}%)")
print(f"False Positives (Pages that did not meet the 20% May decline threshold): {len(false_positives)} ({len(false_positives)/len(top50_df)*100:.1f}%)")

print("\n--- Actual False Positives Extracted from Held-Out Test Set ---")
display(false_positives[["content_hash_id", "gsc_impressions_apr", "gsc_clicks_apr", "gsc_clicks_may", "gsc_avg_position_apr", "predicted_probability"]].head(3))

# Display standardized model coefficients as empirical evidence for feature weights
coef_evidence = pd.DataFrame({
    "Feature": feature_cols,
    "Standardized Coefficient": lr_pipeline.named_steps["model"].coef_[0]
}).sort_values("Standardized Coefficient", key=abs, ascending=False).reset_index(drop=True)

print("\n=== Standardized Logistic Regression Coefficients (Empirical Evidence) ===")
display(coef_evidence)

print("\nFalse Positive Interpretation:")
print("The inspected false positives included pages with relatively high April average-position values. These pages did not meet the May decline label, showing that the static W05 features can flag pages whose subsequent click performance remains above the decline threshold.")

=== W05 Logistic Regression Top-50 Queue Breakdown (Test Set) ===
Total Reviewed: 50
True Positives (Successfully Flagged Declining Pages): 12 (24.0%)
False Positives (Pages that did not meet the 20% May decline threshold): 38 (76.0%)

--- Actual False Positives Extracted from Held-Out Test Set ---


,content_hash_id,gsc_impressions_apr,gsc_clicks_apr,gsc_clicks_may,gsc_avg_position_apr,predicted_probability
14581,content_c6750dfac3e50f22,9120.0,15.0,31.0,37.959863,0.826459
14379,content_9fd0e9e1548758b6,9113.0,18.0,15.0,38.098848,0.783880
14643,content_d1e3ad5bedf64dd6,2926.0,14.0,16.0,32.811457,0.775742



=== Standardized Logistic Regression Coefficients (Empirical Evidence) ===


,Feature,Standardized Coefficient
0,log_ga4_pageviews_feb_apr,-0.939398
1,ga4_data_available,0.706044
2,gsc_avg_position_apr,0.321219
3,log_gsc_clicks_apr,0.280714
4,client_has_ga4,-0.175088
5,log_ga4_engaged_sessions_feb_apr,0.044646
6,log_gsc_impressions_feb_apr,-0.023094
7,client_has_gsc,0.000000
8,gsc_data_available,0.000000



False Positive Interpretation:
The inspected false positives included pages with relatively high April average-position values. These pages did not meet the May decline label, showing that the static W05 features can flag pages whose subsequent click performance remains above the decline threshold.


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.